# Enhancer rebuild — Kaggle dataset audit and grouped split proposal

**One complete Kaggle run.** This notebook does **not** train models. It verifies the original archive, reparses all eight FASTA sources, proves exact overlap, repeats the exhaustive forward/reverse-complement similarity audit, independently checks candidate alignments, reconstructs similarity groups and five-/ten-fold assignments, and exports manifests with audit evidence.

### Upload exactly one ZIP bundle as a private Kaggle Dataset

Upload `Enhancer_Kaggle_Input_Bundle.zip`. It holds exactly two original ZIPs:

1. `DNABERT (Preserve).zip` — unchanged original work.
2. `Enhancer_Sequence_Audit_Checkpoint_02.zip` — previous audit evidence and scripts.

In Kaggle: **Datasets → New Dataset → upload the bundle → Private → Create**. Then import this `.ipynb` as a Kaggle Notebook, and click **Add Input** to attach your dataset. Use **CPU** and leave **Internet off**. Choose **Save Version → Save & Run All**. Outputs appear under `/kaggle/working/enhancer_audit_v1/`; download `enhancer_audit_v1_outputs.zip` from the notebook Output after the run. Files uploaded as notebook content are not automatically in `/kaggle/input`: attach the dataset with *Add Input*. This notebook accepts intact ZIPs and Kaggle's nested extracted directories; it verifies extracted member contents against the original archived file hashes.

**Scope of conclusion:** This run freezes a reproducible *candidate sequence-based* protocol if all checks pass. LCS ≥160 is an intentionally conservative graph rule on 200-bp sequences; it is **not** a formal 80% local identity, whole-genome coordinate, or biological homology guarantee. The supplied legacy independent set has been used previously, so its results remain secondary. Review the output before choosing the final paper protocol and training any model. No additional reference genome or publisher training supplement is required to run this notebook; the outstanding source/coordinate questions are recorded as limitations.


In [2]:
from __future__ import annotations
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import csv, hashlib, io, json, re, shutil, subprocess, sys, tempfile, time
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import sklearn

OUT = Path('/kaggle/working/enhancer_audit_v1') if Path('/kaggle/working').exists() else Path.cwd() / 'enhancer_audit_v1'
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED = {
    'DNABERT (Preserve).zip': '20f59d3b830f93ddae01813ab3a2a4a6a2fe41455062e62b512d3da1dd77ce0c',
    'Enhancer_Sequence_Audit_Checkpoint_02.zip': 'f84392731ecad8b9a6549b927a4207cd731ebbde8525cbd3cd9e1d4c99969399',
}
EXPECTED_MEMBERS = {'DNABERT (Preserve).zip': {'DNABERT (Preserve)/Aditta/DNABERT - Ind Test (Base, BiLSTM).ipynb': '246c2febf94b8f6176f17b7652bab96f2aebef41d0d4129e3118c61d7063750b', 'DNABERT (Preserve)/Aditta/DNABERT - Ind Test.docx': '40bcd75e4936cde91e783cf01f36290b37aecbd9f442c19a1a0a5ae5c0c8926d', 'DNABERT (Preserve)/Aditta/DNABERT- Ind Test (1).ipynb': '12032d608fb06e9442616b42cc28ca2d06756655e46fadaf3dbf69a4da9ff4d7', 'DNABERT (Preserve)/Aditta/DNABERT-10 Fold.docx': '9ce821ac957dad7b33555b57817f03e60aaa9b782617d8364d45b67f64e9f6a7', 'DNABERT (Preserve)/Aditta/DNABERT-10 Fold.ipynb': 'b65599f18c8c96b15945c88c0d2ce3caa0903c66ce9c4fe114ca40931244ecb6', 'DNABERT (Preserve)/Aditta/DNABERT-5 Fold.docx': '24062cf4fcf4d451e66cb611891e80fc2dfd431e06e70a0d9e7e7ace0c464950', 'DNABERT (Preserve)/Aditta/DNABERT-5.ipynb': 'cfec3e1db837ec7f37ec4f8e2d5e0686721a8b7518b3deb80efe12a3d14681cd', 'DNABERT (Preserve)/Arshee/10 fold final.ipynb': 'd50d5e8390224382b85964c53cbdd7fec4f0869a84591cec3b4ea1635fb0d2ec', 'DNABERT (Preserve)/Arshee/10 fold.docx': '2cd740a0daeccf7cd63fb3daed681670fad209017d35ffde1e7697a0be28a7aa', 'DNABERT (Preserve)/Arshee/5 fold final.ipynb': 'c4f5c86d076c086ca02a1053548e88060f4a6e89a54f7a08d18a535bcbde3fbe', 'DNABERT (Preserve)/Arshee/5 fold.docx': 'b1ec7c7be1f7a3b70f81906a3d5058db7a35768ca71a2d4e265c076390c77ce7', 'DNABERT (Preserve)/Arshee/ind test.docx': 'f36dad26530846067830721f7f6d27761d6b0eeb89556ab78492a0fdb48a88e3', 'DNABERT (Preserve)/Arshee/ind-test.ipynb': 'fae63f82432e240895425ead4c4f4a5887e0df3a10af883a1c4afa1545a04896', 'DNABERT (Preserve)/Dataset/enhancer.cv.txt': '5aa1336bc38a675f579244e9dabb3c689abb835880eb42b1ba54451cc667ba61', 'DNABERT (Preserve)/Dataset/enhancer.ind.txt': 'acaea2aca7113947b47d6b672848d07be9cf3cacd7a0c2cf84f554d197cda374', 'DNABERT (Preserve)/Dataset/non.cv.txt': '661abf8105ff64905d9a64b609c4112af637ee44b3157ad8742f03960ded3412', 'DNABERT (Preserve)/Dataset/non.ind.txt': 'deb9f732de90814b0b80bf48e224778746765eb8915a5c0f4501de8510eec3fa', 'DNABERT (Preserve)/Dataset/strong_100.txt': '0b808bfa688475c65b0f61aa5f4bb38518369501cb518f6fe0a18093957a250d', 'DNABERT (Preserve)/Dataset/strong_742.txt': '1e19da34bec49d5425faaa1841feb6086e2c306f20f1ffca54e262f61e058dcb', 'DNABERT (Preserve)/Dataset/weak_100.txt': '0d8a382444797696afb45f1484db9aa5393acaac11f931eb7f0b9f292871a6b9', 'DNABERT (Preserve)/Dataset/weak_742.txt': 'c9a74473bbf5001e3ef123c889bffe814b610baa15b80d18b1112b07ca857bf9', 'DNABERT (Preserve)/Paper Draft Final-2.docx': '28a2dbda7332fa068d652d446569cb0c363a0502e7bc26745c38ca9e0e50e57d'}, 'Enhancer_Sequence_Audit_Checkpoint_02.zip': {'Enhancer_Audit_Checkpoint_02.md': '32e50645a7d5a96902c858a89556b6333ced9ad7373cad0e5f9cfaadedb53633', 'README.md': '78d0ff8eb1cf841cc3779e2b030397ae2feb5cb3bfa34f2f5c98a6e7b47ed479', 'scripts/sequence_lcs_audit.cpp': 'c3e7045874107676d04181f5833d98020c35e2325ecc5c96a59954ffee1634b5', 'scripts/prepare_lcs_input.py': '88685cb731aa8ef82e36d6dba8b388f1d78d4fbcc118abf124cf324734c474d6', 'scripts/gapped_similarity_audit.py': '0eaa0a46190fe87d6b648875484621eee00689ad3c269a39b04b00aaad017e15', 'scripts/propose_similarity_repair.py': 'f7c845b5bf91e5496f87f3f4db3aeae902013832f9720b3e756521204d7b9a64', 'source/stage1_records.csv': 'e47682387ef16ac5fad3981a59a1cbfbfb182866317a0c2aaec31483122a9a5c', 'source/stage2_records.csv': '347b3c070ec3e8907c6fe5241b788e6f16dcd763b5a86be9032879aaebb4e2ae', 'source/audit_summary.json': '7f188939d23d6acd3582759c95b47605265decd980e5312be03ddb5c3f2abbff', 'source/publisher_s4_metadata_400.tsv': 'bc3ad20d91ece0cf753c5bad95dd7e833a5c07052aec73b6cf2861b91b62ea9c', 'results/stage1_lcs_ge150.tsv': 'f8eac25b13aa54ea4ca6115f4a8813fbd10345cce2321fc8b79df8415bffac21', 'results/stage1_gapped_ge160_candidates.csv': 'e7eef3b6b81d6823176d40441ca166d983a0a15c09c32d136774893b9d64456e', 'proposal/stage1_similarity_proposal.csv': 'c0f69b07f0174008759c919adc3489c87841892820473b6b33fabf346a4f6432', 'proposal/stage2_similarity_proposal.csv': 'd35a0db6e70ec99ecdfd94d353fbae07bb317a829e6c710466ed421cf36faefc', 'proposal/similarity_proposal_summary.json': 'd92681347ada5f999196d9e95c207a7d2b67196c0ab69e6baf67b4f6b73276b0'}}
def digest(data): return hashlib.sha256(data).hexdigest()
def file_sha(path): return digest(Path(path).read_bytes())
def find_input(filename):
    roots = [Path('/kaggle/input'), Path.cwd(), Path.cwd()/'preserved_download']
    found = sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(filename) if p.is_file()})
    if not found:
        bundles=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob('Enhancer_Kaggle_Input_Bundle.zip') if p.is_file()})
        if len(bundles)==1:
            with ZipFile(bundles[0]) as z:
                assert set(z.namelist())==set(EXPECTED), 'Unexpected bundle contents'
                assert z.getinfo(filename).file_size < 32*1024*1024
                nested=OUT/'bundle_input'/filename
                nested.parent.mkdir(parents=True,exist_ok=True)
                nested.write_bytes(z.read(filename))
            found=[nested]
    if not found:
        stem=Path(filename).stem
        def rel(member):
            return member[len(stem)+1:] if member.startswith(stem+'/') else member
        found=sorted({d.resolve() for root in roots if root.exists()
                      for d in root.rglob(stem) if d.is_dir()
                      and all((d/rel(member)).is_file() for member in EXPECTED_MEMBERS[filename])})
    if len(found) != 1:
        raise RuntimeError(f'Expected exactly one {filename} among attached inputs; found: {found}. Attach the dataset with Enhancer_Kaggle_Input_Bundle.zip.')
    return found[0]
inputs = {name: find_input(name) for name in EXPECTED}
class ExtractedArchive:
    def __init__(self, path):self.path=Path(path);self.name=self.path.name+'.zip'
    def __enter__(self):return self
    def __exit__(self,*args):return False
    def namelist(self):return list(EXPECTED_MEMBERS[self.name])
    def read(self,member):
        assert member in EXPECTED_MEMBERS[self.name], f'Unexpected extracted member {member}'
        prefix=self.name.removesuffix('.zip')+'/'
        relative=member[len(prefix):] if member.startswith(prefix) else member
        data=(self.path/relative).read_bytes()
        assert digest(data)==EXPECTED_MEMBERS[self.name][member], f'Expanded member hash mismatch: {member}'
        return data
    def testzip(self):
        for member in self.namelist():self.read(member)
        return None
def archive(path):return ExtractedArchive(path) if Path(path).is_dir() else ZipFile(path)
for name, path in inputs.items():
    if path.is_file():
        got = file_sha(path)
        assert got == EXPECTED[name], f'{name}: SHA-256 differs; use the unchanged shared archive. Got {got}'
    with archive(path) as z:
        assert z.testzip() is None, f'{name}: ZIP corruption'
        if path.is_dir():
            assert set(z.namelist())==set(EXPECTED_MEMBERS[name])
            print(f'INPUT OK: {name} ({path}, verified {len(z.namelist())} extracted files against original member hashes; ZIP byte hash unavailable after Kaggle extraction)')
        else:print(f'INPUT OK: {name} ({path}, sha256={got})')
print('pandas', pd.__version__, 'sklearn', sklearn.__version__)

INPUT OK: DNABERT (Preserve).zip (/kaggle/input/datasets/aryanaditta/enhancer-kaggle-input-bundle-zip/DNABERT (Preserve)/DNABERT (Preserve), verified 22 extracted files against original member hashes; ZIP byte hash unavailable after Kaggle extraction)
INPUT OK: Enhancer_Sequence_Audit_Checkpoint_02.zip (/kaggle/input/datasets/aryanaditta/enhancer-kaggle-input-bundle-zip/Enhancer_Sequence_Audit_Checkpoint_02, verified 15 extracted files against original member hashes; ZIP byte hash unavailable after Kaggle extraction)
pandas 2.3.3 sklearn 1.6.1


## 1. Parse original sequences and verify the archive evidence

Every record must contain exactly 200 A/C/G/T bases. The six label files are matched against the archived Stage 1/2 rows by stable sequence digest and source; the combined enhancer files are checked separately. Original files remain untouched.

In [3]:
FILES = ['enhancer.cv.txt','enhancer.ind.txt','non.cv.txt','non.ind.txt',
         'strong_742.txt','strong_100.txt','weak_742.txt','weak_100.txt']
def parse_fasta(raw, name):
    result, head, pieces = [], None, []
    def save():
        if head is None: return
        seq = ''.join(pieces).upper()
        assert re.fullmatch('[ACGT]{200}', seq), f'{name}: invalid sequence at {head}'
        result.append(dict(source_file=name, header=head, sequence=seq,
                           record_id=digest(seq.encode('ascii'))[:16], sha256=digest(seq.encode('ascii'))))
    for line in raw.decode('utf-8-sig').splitlines():
        line=line.strip()
        if not line: continue
        if line.startswith('>'):
            save(); head=line[1:]; pieces=[]
        else:
            assert head is not None, f'{name}: sequence without header'
            pieces.append(line)
    save()
    assert len(result)==len({r['sha256'] for r in result}) and result, f'{name}: empty or internally duplicate'
    return result

with archive(inputs['DNABERT (Preserve).zip']) as rawzip:
    paths={Path(p).name: p for p in rawzip.namelist() if Path(p).name in FILES}
    assert len(paths)==len(FILES), f'Missing FASTA names: {set(FILES)-set(paths)}'
    original={name:parse_fasta(rawzip.read(paths[name]), name) for name in FILES}
with archive(inputs['Enhancer_Sequence_Audit_Checkpoint_02.zip']) as auditzip:
    source_rows={s:pd.read_csv(io.BytesIO(auditzip.read(f'source/stage{s}_records.csv')), dtype=str, keep_default_na=False)
                 for s in (1,2)}
    archived_pairs=pd.read_csv(io.BytesIO(auditzip.read('results/stage1_lcs_ge150.tsv')),sep='\t',dtype=str,keep_default_na=False)
    archived_proposal={s:pd.read_csv(io.BytesIO(auditzip.read(f'proposal/stage{s}_similarity_proposal.csv')),dtype=str,keep_default_na=False)
                       for s in (1,2)}
    archived_summary=json.loads(auditzip.read('proposal/similarity_proposal_summary.json'))
    publisher=pd.read_csv(io.BytesIO(auditzip.read('source/publisher_s4_metadata_400.tsv')),sep='\t',dtype=str,keep_default_na=False)
    assert len(publisher)==400 and Counter(publisher.subtype)=={'strong':100,'weak':100,'non':200}

def idset(name):return {r['sha256'] for r in original[name]}
assert {n:len(original[n]) for n in FILES} == {'enhancer.cv.txt':1484,'enhancer.ind.txt':200,'non.cv.txt':1484,'non.ind.txt':200,
  'strong_742.txt':742,'strong_100.txt':100,'weak_742.txt':742,'weak_100.txt':100}
assert idset('enhancer.cv.txt') == idset('strong_742.txt') | idset('weak_742.txt')
assert idset('enhancer.ind.txt') == idset('strong_100.txt') | idset('weak_100.txt')
assert idset('strong_100.txt') <= idset('strong_742.txt')
assert len(idset('strong_100.txt') & idset('strong_742.txt'))==100
sources=['strong_742.txt','weak_742.txt','non.cv.txt','strong_100.txt','weak_100.txt','non.ind.txt']
for i,a in enumerate(sources):
    for b in sources[i+1:]:
        overlap = idset(a)&idset(b)
        assert not overlap or {a,b}=={'strong_742.txt','strong_100.txt'}, f'Unexpected exact overlap: {a}, {b}, {len(overlap)}'
print('Raw counts:', {n:len(original[n]) for n in FILES})
print('Verified: all 100 legacy independent strong sequences appear in the supplied training strong file.')

Raw counts: {'enhancer.cv.txt': 1484, 'enhancer.ind.txt': 200, 'non.cv.txt': 1484, 'non.ind.txt': 200, 'strong_742.txt': 742, 'strong_100.txt': 100, 'weak_742.txt': 742, 'weak_100.txt': 100}
Verified: all 100 legacy independent strong sequences appear in the supplied training strong file.


In [4]:
# Rebuild from the six raw label sources, then compare every row to the independently archived parse.
row_fields=['record_id','stage','split','label','subtype','source_file','header','sequence','sha256']
layouts={1:[('strong_742.txt','train','1','strong'),('weak_742.txt','train','1','weak'),
            ('non.cv.txt','train','0','non'),('strong_100.txt','test','1','strong'),
            ('weak_100.txt','test','1','weak'),('non.ind.txt','test','0','non')],
         2:[('strong_742.txt','train','1','strong'),('weak_742.txt','train','0','weak'),
            ('strong_100.txt','test','1','strong'),('weak_100.txt','test','0','weak')]}
rebuilt={}
for stage, blocks in layouts.items():
    items=[]
    for source,split,label,subtype in blocks:
        for r in original[source]:
            if split=='train' and r['sha256'] in idset('strong_100.txt'): continue
            items.append({**r,'stage':str(stage),'split':split,'label':label,'subtype':subtype})
    df=pd.DataFrame(items)
    old=source_rows[stage]
    assert len(df)==len(old) and df[row_fields].equals(old[row_fields]), f'Stage {stage} differs from audited multiline FASTA parsing'
    assert df.record_id.is_unique
    assert not (set(df.loc[df.split=='train','sha256']) & set(df.loc[df.split=='test','sha256']))
    rebuilt[stage]=df
    print(f'Stage {stage}: training {sum(df.split=="train")}, historical independent test {sum(df.split=="test")}, total {len(df)}')
assert set(rebuilt[2].record_id) == set(rebuilt[1].loc[rebuilt[1].subtype.isin(['strong','weak']),'record_id'])

Stage 1: training 2868, historical independent test 400, total 3268
Stage 2: training 1384, historical independent test 200, total 1584


## 2. Repeat exhaustive similarity screening

The bundled C++ program tests **every** unordered Stage 1 pair, in both orientations, for exact longest common subsequence (LCS). It saves every pair with LCS ≥150. The archived result is compared to the recomputed result; if the output differs, the run stops.

In [5]:
with archive(inputs['Enhancer_Sequence_Audit_Checkpoint_02.zip']) as z:
    cpp = z.read('scripts/sequence_lcs_audit.cpp')
cpp_path=OUT/'sequence_lcs_audit.cpp'; cpp_path.write_bytes(cpp)
assert shutil.which('g++'), 'Kaggle CPU environment requires g++ for the exhaustive similarity check.'
binary=OUT/'sequence_lcs_audit'
subprocess.run(['g++','-O3','-std=c++17',str(cpp_path),'-o',str(binary)],check=True)
lcs_input=OUT/'stage1_lcs_input.tsv'
with lcs_input.open('w') as f:
    for r in source_rows[1].to_dict('records'):
        f.write('\t'.join([r['record_id'],r['split'],r['fold_5'] or 'NA',r['fold_10'] or 'NA',r['subtype'],r['sequence']])+'\n')
start=time.time()
full_path=OUT/'stage1_lcs_ge150_recomputed.tsv'
run=subprocess.run([str(binary),str(lcs_input),str(full_path)],capture_output=True,text=True,check=True,timeout=1800)
print(run.stderr[-900:]); print('Screen elapsed seconds:',round(time.time()-start,1))
pairs=pd.read_csv(full_path,sep='\t',dtype=str,keep_default_na=False)
assert pairs.equals(archived_pairs), 'Full similarity screen differs from the archived candidates.'
high=pairs[pairs[['lcs_forward','lcs_rc']].astype(int).max(axis=1)>=160].copy()
assert len(rebuilt[1])==3268 and len(pairs)==425 and len(high)==75
print('Every distinct pair checked:',len(rebuilt[1])*(len(rebuilt[1])-1)//2)
print('LCS >=150:',len(pairs),'LCS >=160:',len(high),'identical to previous run:',pairs.equals(archived_pairs))

processed 0 / 3268
processed 500 / 3268
processed 1000 / 3268
processed 1500 / 3268
processed 2000 / 3268
processed 2500 / 3268
processed 3000 / 3268
total=5338278 max=188 >=128=1418915 >=140=2667 >=150=425 >=160=75 cross_test_max=172 cross_fold5_max=188 cross_fold10_max=188

Screen elapsed seconds: 52.6
Every distinct pair checked: 5338278
LCS >=150: 425 LCS >=160: 75 identical to previous run: True


## 3. Independent affine-gap alignment of all high-LCS candidates

This is a sensitivity check, not a replacement for grouping: all 75 candidates stay grouped regardless of this aligner's score. Gap penalties and the scored alignment are preserved in the outputs.

In [6]:
with archive(inputs['Enhancer_Sequence_Audit_Checkpoint_02.zip']) as z:
    align_script=OUT/'gapped_similarity_audit.py'; align_script.write_bytes(z.read('scripts/gapped_similarity_audit.py'))
    (OUT/'stage1_records_audit.csv').write_bytes(z.read('source/stage1_records.csv'))
align_path=OUT/'stage1_gapped_ge160_recomputed.csv'
aligned=subprocess.run([sys.executable,str(align_script),'--records',str(OUT/'stage1_records_audit.csv'),
                        '--candidates',str(full_path),'--output',str(align_path)],capture_output=True,text=True,check=True,timeout=600)
print(aligned.stdout.strip())
alignment=pd.read_csv(align_path)
with archive(inputs['Enhancer_Sequence_Audit_Checkpoint_02.zip']) as z:
    previous_alignment=pd.read_csv(io.BytesIO(z.read('results/stage1_gapped_ge160_candidates.csv')))
assert len(alignment)==len(previous_alignment)==75
assert alignment.fillna('').astype(str).equals(previous_alignment.fillna('').astype(str)), 'Candidate alignments differ from archived check'
print('Optimal global alignment with >=160 matches:',int((alignment.matches>=160).sum()))

{'80pct_pairs': 40, 'cross_train_test': 11, 'cross_fold5_train': 24, 'cross_fold10_train': 24, 'cross_stage1_label': 10}
Optimal global alignment with >=160 matches: 40


## 4. Build and verify grouped fold proposal

The bundled grouping script unions all 75 Stage 1 LCS ≥160 pairs, excludes candidate training records sharing a group with the historical test, and assigns grouped 5- and 10-fold validation to retained training records. Stage 2 reuses the corresponding Stage 1 group IDs. Fold assignments use the installed Kaggle scikit-learn version, which is recorded below.

In [7]:
with archive(inputs['Enhancer_Sequence_Audit_Checkpoint_02.zip']) as z:
    script=OUT/'propose_similarity_repair.py'; script.write_bytes(z.read('scripts/propose_similarity_repair.py'))
    for s in (1,2):
        (OUT/f'stage{s}_records_audit.csv').write_bytes(z.read(f'source/stage{s}_records.csv'))
proposal_dir=OUT/'recomputed_proposal'
cmd=[sys.executable,str(script),'--stage1',str(OUT/'stage1_records_audit.csv'),
     '--stage2',str(OUT/'stage2_records_audit.csv'),'--lcs-candidates',str(full_path),
     '--output',str(proposal_dir)]
subprocess.run(cmd,capture_output=True,text=True,check=True,timeout=300)
proposed={s:pd.read_csv(proposal_dir/f'stage{s}_similarity_proposal.csv',dtype=str,keep_default_na=False) for s in (1,2)}
for stage, p in proposed.items():
    src=rebuilt[stage]
    assert p.record_id.equals(src.record_id)
    assert p.sequence_sha256.equals(src.sha256)
    assert p.label.equals(src.label) and p.subtype.equals(src.subtype)
    assert p.candidate_role.equals(archived_proposal[stage].candidate_role)
    assert p.similarity_group_id.equals(archived_proposal[stage].similarity_group_id)
    want={1:{'train_candidate':2838,'legacy_test':400,'excluded_related_to_legacy_test':30},
          2:{'train_candidate':1376,'legacy_test':200,'excluded_related_to_legacy_test':8}}[stage]
    assert p.candidate_role.value_counts().to_dict()==want
    assert set(p.loc[p.candidate_role=='legacy_test','similarity_group_id']).isdisjoint(
        set(p.loc[p.candidate_role=='train_candidate','similarity_group_id']))
    for k in (5,10):
        f=f'candidate_fold_{k}'
        train=p[p.candidate_role=='train_candidate']
        assert set(train[f])==set(map(str,range(k)))
        assert p.loc[p.candidate_role!='train_candidate',f].eq('').all()
        assert train.groupby('similarity_group_id')[f].nunique().max()==1
    print('Stage',stage,'roles',want,'5-fold sizes',p.loc[p.candidate_role=='train_candidate','candidate_fold_5'].value_counts().sort_index().to_dict())
    for k in (5,10):
        f=f'candidate_fold_{k}'
        diff=((p[f] != archived_proposal[stage][f]) & (p.candidate_role=='train_candidate')).sum()
        print(f'  {k}-fold assignment changes from archived proposal:',int(diff))
# Explicit Stage 1 / Stage 2 shared group comparison independent of dataframe ordering.
groups1=proposed[1].set_index('record_id').similarity_group_id.to_dict()
assert all(groups1[r.record_id]==r.similarity_group_id for r in proposed[2].itertuples())
for stage in (1,2):
    lookup=proposed[stage].set_index('record_id')
    stage_pairs=high if stage==1 else high[high.id_a.isin(lookup.index)&high.id_b.isin(lookup.index)]
    for a,b in stage_pairs[['id_a','id_b']].itertuples(index=False,name=None):
        assert lookup.loc[a,'similarity_group_id']==lookup.loc[b,'similarity_group_id']
        assert not ({lookup.loc[a,'candidate_role'],lookup.loc[b,'candidate_role']}=={'train_candidate','legacy_test'})
    print('Stage',stage,'screen-positive pairs kept together:',len(stage_pairs))

Stage 1 roles {'train_candidate': 2838, 'legacy_test': 400, 'excluded_related_to_legacy_test': 30} 5-fold sizes {'0': 571, '1': 566, '2': 567, '3': 563, '4': 571}
  5-fold assignment changes from archived proposal: 2281
  10-fold assignment changes from archived proposal: 2563
Stage 2 roles {'train_candidate': 1376, 'legacy_test': 200, 'excluded_related_to_legacy_test': 8} 5-fold sizes {'0': 273, '1': 274, '2': 276, '3': 276, '4': 277}
  5-fold assignment changes from archived proposal: 1109
  10-fold assignment changes from archived proposal: 1250
Stage 1 screen-positive pairs kept together: 75
Stage 2 screen-positive pairs kept together: 12


## 5. Export complete candidate manifests and a readable QA report

Use `candidate_role == train_candidate` and `fold_5` for the first model comparison. Validation predictions belong only to the held-out fold. Reserve `legacy_test` for a secondary historical comparison after models and choices are fixed. Excluded records stay documented and are never silently relabeled. Perform any reverse-complement augmentation **only after** grouping and within the training part of each fold.

In [8]:
outputs={}
for stage in (1,2):
    p=proposed[stage]
    raw=rebuilt[stage]
    result=pd.DataFrame({
        'record_id':raw.record_id,'sequence_sha256':raw.sha256,'stage':raw.stage,
        'sequence':raw.sequence,'label':raw.label,'subtype':raw.subtype,
        'source_file':raw.source_file,'header':raw.header,'original_split':raw.split,
        'role':p.candidate_role,'similarity_group_id':p.similarity_group_id,
        'fold_5':p.candidate_fold_5,'fold_10':p.candidate_fold_10,
    })
    assert result.record_id.is_unique
    assert result.sequence.map(len).eq(200).all()
    assert (result.sequence.map(lambda s:digest(s.encode('ascii')))==result.sequence_sha256).all()
    # Ensure exact sequence and reverse complement cannot appear across retained partitions.
    rc = str.maketrans('ACGT','TGCA')
    train=result[result.role=='train_candidate']; test=result[result.role=='legacy_test']
    tset=set(test.sequence)
    assert all(s not in tset and s.translate(rc)[::-1] not in tset for s in train.sequence)
    for k in (5,10):
        seq_to_fold=dict(zip(train.sequence,train[f'fold_{k}']))
        assert all(seq_to_fold.get(s.translate(rc)[::-1],fold)==fold for s,fold in seq_to_fold.items())
    name=f'stage{stage}_candidate_manifest.csv'
    result.to_csv(OUT/name,index=False)
    outputs[name]=file_sha(OUT/name)

mix=proposed[1].groupby('similarity_group_id').label.nunique()
conflict_ids=set(mix[mix>1].index)
conflicts=proposed[1][proposed[1].similarity_group_id.isin(conflict_ids)]
conflicts.to_csv(OUT/'mixed_label_groups_review.csv',index=False)
outputs['mixed_label_groups_review.csv']=file_sha(OUT/'mixed_label_groups_review.csv')
publisher.to_csv(OUT/'publisher_s4_metadata_400.tsv',sep='\t',index=False)
outputs['publisher_s4_metadata_400.tsv']=file_sha(OUT/'publisher_s4_metadata_400.tsv')
for name in ('stage1_lcs_ge150_recomputed.tsv','stage1_gapped_ge160_recomputed.csv'):
    outputs[name]=file_sha(OUT/name)

summary={
 'status':'CANDIDATE_SEQUENCE_LCS160_PROTOCOL_REVIEW_REQUIRED',
 'inputs':{n:{'expected_original_zip_sha256':EXPECTED[n],
              'verification_mode':'zip_sha256' if inputs[n].is_file() else 'all_extracted_member_sha256',
              'verified_member_count':len(EXPECTED_MEMBERS[n])} for n in inputs},
 'rules':{'length_bp':200,'pair_graph':'exact LCS >=160 in either orientation, on all distinct stage-1 pairs',
          'test_related':'exclude training records in a graph component containing a historical test record',
          'cv':'StratifiedGroupKFold seed=42; 5 and 10 folds; augmentation after split'},
 'stages':{str(s):{'roles':proposed[s].candidate_role.value_counts().to_dict(),
                   'label_counts_by_role':pd.crosstab(proposed[s].candidate_role,proposed[s].label).to_dict(),
                   'fold_5_counts':proposed[s].query("candidate_role == 'train_candidate'").candidate_fold_5.value_counts().sort_index().to_dict()}
            for s in (1,2)},
 'pair_screen':{'total_stage1_pairs':5338278,'lcs_ge150':len(pairs),'lcs_ge160':len(high),
                'optimal_global_alignment_matches_ge160':int((alignment.matches>=160).sum()),
                'mixed_stage1_label_groups':len(conflict_ids)},
 'environment':{'python':sys.version,'scikit_learn':sklearn.__version__,'pandas':pd.__version__},
 'limitations':['The LCS cutoff is a conservative graph rule, not an 80% local sequence-identity threshold.',
                'No genome mapping or chromosome-disjoint assessment was performed.',
                'Source of the discrepancy between the supplied training file and publisher independent-set claim is unresolved.',
                'The historical independent set has been examined previously and is secondary; never describe it as untouched.',
                'This run produces no model performance claims.'],
 'outputs_sha256':outputs,
}
(OUT/'audit_run_summary.json').write_text(json.dumps(summary,indent=2,sort_keys=True)+'\n')
readme='''# Enhancer candidate protocol output

STATUS: CANDIDATE_SEQUENCE_LCS160_PROTOCOL_REVIEW_REQUIRED

Use stage1_candidate_manifest.csv and stage2_candidate_manifest.csv.
Main comparison: training rows with role=train_candidate; 5-fold grouped CV; fold_5 is validation fold number 0..4. stage1 label: enhancer=1, non=0; stage2 label: strong=1, weak=0. Fold assignments stay fixed across models. No augmentation before splitting. Train/validation separation must occur BEFORE any supervised tuning, fitting, calibration, vocabulary fitting, feature selection, or preprocessing estimation. Tuning requires inner folds derived only from the current training fold.

Historical 400/200-record legacy_test is secondary; excluded_related_to_legacy_test rows cannot train in the historical-test analysis. The folds are a candidate sequence-based protocol pending review; genomic coordinates and publisher source conflict are unresolved. Inspect mixed_label_groups_review.csv. Record all changes as a new version rather than modifying these files silently.

See audit_run_summary.json for input hashes, counts, run versions, limits, and output hashes. No performance metric is produced here.
'''
(OUT/'README_USE_FOR_MODELING.txt').write_text(readme)
print('EXPORT SUMMARY')
print(json.dumps({k:summary[k] for k in ['status','stages','pair_screen','limitations']},indent=2))
print('Candidate files:',OUT/'stage1_candidate_manifest.csv',OUT/'stage2_candidate_manifest.csv')

EXPORT SUMMARY
{
  "status": "CANDIDATE_SEQUENCE_LCS160_PROTOCOL_REVIEW_REQUIRED",
  "stages": {
    "1": {
      "roles": {
        "train_candidate": 2838,
        "legacy_test": 400,
        "excluded_related_to_legacy_test": 30
      },
      "label_counts_by_role": {
        "0": {
          "excluded_related_to_legacy_test": 22,
          "legacy_test": 200,
          "train_candidate": 1462
        },
        "1": {
          "excluded_related_to_legacy_test": 8,
          "legacy_test": 200,
          "train_candidate": 1376
        }
      },
      "fold_5_counts": {
        "0": 571,
        "1": 566,
        "2": 567,
        "3": 563,
        "4": 571
      }
    },
    "2": {
      "roles": {
        "train_candidate": 1376,
        "legacy_test": 200,
        "excluded_related_to_legacy_test": 8
      },
      "label_counts_by_role": {
        "0": {
          "excluded_related_to_legacy_test": 7,
          "legacy_test": 100,
          "train_candidate": 735
        },
 

In [9]:
# Zip the portable handoff; omit executable and source input copies to keep download small.
zip_path=OUT.parent/'enhancer_audit_v1_outputs.zip'
keep=['README_USE_FOR_MODELING.txt','audit_run_summary.json','stage1_candidate_manifest.csv',
      'stage2_candidate_manifest.csv','mixed_label_groups_review.csv','publisher_s4_metadata_400.tsv',
      'stage1_lcs_ge150_recomputed.tsv','stage1_gapped_ge160_recomputed.csv']
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED) as z:
    for name in keep:z.write(OUT/name,arcname=name)
with ZipFile(zip_path) as z:
    assert z.testzip() is None
    assert sorted(z.namelist())==sorted(keep)
print('Download this Kaggle output:',zip_path)
print('sha256:',file_sha(zip_path),'bytes:',zip_path.stat().st_size)

Download this Kaggle output: /kaggle/working/enhancer_audit_v1_outputs.zip
sha256: 72aaf60b40304dcf5007f7f4c839ddbf032385e9b7a99fb2d5fc652a3b33fa7c bytes: 668593
